# Flow Around a Circular Cylinder — Potential Flow Theory

Classical inviscid, incompressible potential flow past a circular cylinder: uniform flow superposed with a doublet (non-lifting case), then a point vortex added to model circulation and lift (the Magnus effect). Validated against two textbook results: d'Alembert's paradox (zero drag) and the Kutta-Joukowski theorem ($L' = \rho U_\infty \Gamma$).

For the real, viscous counterpart — boundary-layer separation, wake formation, vortex shedding, and the drag potential flow theory cannot predict — see the companion CFD notebook at `CFD/CFD Basics/cylinder_cfd/main.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Non-lifting flow: uniform flow + doublet

A uniform flow $U_\infty$ in the $+x$ direction, superposed with a doublet of strength $\kappa = 2\pi U_\infty R^2$ at the origin, produces flow around a cylinder of radius $R$. In polar coordinates:

$$\psi(r,\theta) = U_\infty \sin\theta \left(r - \frac{R^2}{r}\right)$$

$$V_r = \frac{1}{r}\frac{\partial \psi}{\partial \theta} = U_\infty \cos\theta \left(1 - \frac{R^2}{r^2}\right), \qquad V_\theta = -\frac{\partial \psi}{\partial r} = -U_\infty \sin\theta \left(1 + \frac{R^2}{r^2}\right)$$

On the surface ($r=R$): $V_r = 0$ (no penetration, confirming $r=R$ is indeed a streamline) and $V_\theta = -2U_\infty \sin\theta$.

In [ ]:
U_inf = 1.0
R = 1.0


def velocity_field(X, Y, Gamma=0.0):
    """Cartesian velocity field: uniform flow + doublet + point vortex (clockwise-positive Gamma)."""
    r = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)
    Vr = U_inf * np.cos(theta) * (1 - R**2 / r**2)
    Vtheta = -U_inf * np.sin(theta) * (1 + R**2 / r**2) - Gamma / (2 * np.pi * r)
    Vx = Vr * np.cos(theta) - Vtheta * np.sin(theta)
    Vy = Vr * np.sin(theta) + Vtheta * np.cos(theta)
    inside = r < R
    return np.where(inside, np.nan, Vx), np.where(inside, np.nan, Vy)


def plot_streamlines(Gamma, title, ax=None):
    x = np.linspace(-3, 3, 400)
    y = np.linspace(-3, 3, 400)
    X, Y = np.meshgrid(x, y)
    Vx, Vy = velocity_field(X, Y, Gamma)

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ax.streamplot(X, Y, Vx, Vy, density=1.6, color="steelblue", linewidth=0.7, arrowsize=0.8)
    ax.add_patch(plt.Circle((0, 0), R, color="black", zorder=5))
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect("equal")
    ax.set_title(title)
    return ax


plot_streamlines(0.0, "Non-lifting flow ($\\Gamma=0$)")
plt.show()

## 2. Surface pressure distribution

From Bernoulli's equation (steady, incompressible, along a streamline from the undisturbed freestream), the pressure coefficient depends only on the local surface speed:

$$C_p = 1 - \left(\frac{V_\theta}{U_\infty}\right)^2 = 1 - 4\sin^2\theta$$

This predicts stagnation points ($C_p=1$) at the leading and trailing edges ($\theta=0,\pi$) and the minimum pressure ($C_p=-3$) at the shoulders ($\theta=\pm 90°$) — both checked numerically below.

In [ ]:
def cp_surface(theta, Gamma=0.0):
    Vtheta = -2 * U_inf * np.sin(theta) - Gamma / (2 * np.pi * R)
    return 1 - (Vtheta / U_inf) ** 2


theta = np.linspace(0, 2 * np.pi, 1000, endpoint=False)
cp0 = cp_surface(theta, Gamma=0.0)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(np.degrees(theta), cp0)
ax.invert_yaxis()
ax.set_xlabel("theta (deg)")
ax.set_ylabel("$C_p$")
ax.set_title("Non-lifting cylinder surface $C_p$")
ax.grid(alpha=0.3)
plt.show()

print(f"min Cp = {cp0.min():.4f} (theory: -3.0000)")
print(f"Cp at theta=0 (deg)  = {cp_surface(np.array([0.0]))[0]:.4f} (theory: 1.0000, stagnation)")
print(f"Cp at theta=180 (deg) = {cp_surface(np.array([np.pi]))[0]:.4f} (theory: 1.0000, stagnation)")

## 3. Lifting flow: added circulation (Magnus effect)

Superposing a point vortex of strength $\Gamma$ (defined here **clockwise-positive**, so positive $\Gamma$ produces positive/upward lift) adds a term to the tangential velocity:

$$V_\theta(R,\theta) = -2U_\infty \sin\theta - \frac{\Gamma}{2\pi R}$$

Stagnation points ($V_\theta=0$) occur where $\sin\theta = -\Gamma/(4\pi R U_\infty)$. Depending on the circulation strength relative to $4\pi R U_\infty$, the flow falls into three regimes:

- **Subcritical** ($|\Gamma| < 4\pi R U_\infty$): two stagnation points on the cylinder surface, shifted from $\theta=0,\pi$.
- **Critical** ($|\Gamma| = 4\pi R U_\infty$): the two stagnation points merge into one.
- **Supercritical** ($|\Gamma| > 4\pi R U_\infty$): the stagnation point lifts off the surface entirely.

In [ ]:
Gamma_crit = 4 * np.pi * R * U_inf

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plot_streamlines(0.5 * Gamma_crit, "Subcritical\n$\\Gamma = 0.5\\Gamma_{crit}$", ax=axes[0])
plot_streamlines(Gamma_crit, "Critical\n$\\Gamma = \\Gamma_{crit}$", ax=axes[1])
plot_streamlines(1.5 * Gamma_crit, "Supercritical\n$\\Gamma = 1.5\\Gamma_{crit}$", ax=axes[2])
plt.tight_layout()
plt.show()

## 4. Force integration: d'Alembert's paradox and the Kutta-Joukowski theorem

Integrating the surface pressure around the cylinder gives the force coefficients (normalized by diameter $D=2R$, per unit span):

$$C_d = -\frac{1}{2}\int_0^{2\pi} C_p \cos\theta \, d\theta, \qquad C_l = -\frac{1}{2}\int_0^{2\pi} C_p \sin\theta \, d\theta$$

Potential flow theory predicts **zero drag regardless of circulation** — d'Alembert's paradox, a direct consequence of the flow having no viscosity and therefore no mechanism to dissipate energy into a wake. It also predicts lift matching the Kutta-Joukowski theorem exactly: $L' = \rho U_\infty \Gamma$, or in coefficient form, $C_l = \Gamma/(R U_\infty)$.

In [ ]:
def force_coefficients(Gamma):
    theta = np.linspace(0, 2 * np.pi, 200_000, endpoint=False)
    dtheta = theta[1] - theta[0]
    cp = cp_surface(theta, Gamma)
    Cd = -0.5 * np.sum(cp * np.cos(theta)) * dtheta
    Cl = -0.5 * np.sum(cp * np.sin(theta)) * dtheta
    return Cd, Cl


print(f"{'Gamma':>8} {'Cd (numerical)':>16} {'Cl (numerical)':>16} {'Cl (Kutta-Joukowski)':>22}")
for Gamma in [0.0, 2.0, 5.0, 8.0]:
    Cd, Cl = force_coefficients(Gamma)
    Cl_kj = Gamma / (R * U_inf)
    print(f"{Gamma:8.2f} {Cd:16.6f} {Cl:16.6f} {Cl_kj:22.6f}")

## 5. Circulation sensitivity: $C_l$ vs $\Gamma$

Sweeping the circulation confirms the linear relationship predicted by Kutta-Joukowski holds across the full range, including past the critical circulation where the stagnation point leaves the surface.

In [ ]:
Gammas = np.linspace(0, 2 * Gamma_crit, 40)
Cls = [force_coefficients(g)[1] for g in Gammas]
Cls_kj = Gammas / (R * U_inf)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(Gammas, Cls, "o", ms=4, label="numerical (surface $C_p$ integration)")
ax.plot(Gammas, Cls_kj, "-", label="Kutta-Joukowski, $C_l=\\Gamma/(RU_\\infty)$")
ax.axvline(Gamma_crit, color="gray", ls=":", lw=1, label="$\\Gamma_{crit}$")
ax.set_xlabel("$\\Gamma$")
ax.set_ylabel("$C_l$")
ax.set_title("Lift coefficient vs. circulation")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## 6. Where potential flow breaks down

Real (viscous) flow past a cylinder separates from the surface well before reaching the trailing edge — typically around $\theta \approx 80°$ from the front stagnation point at moderate Reynolds numbers — because the adverse pressure gradient in the aft portion of the potential-flow solution is more than the boundary layer can withstand. Past separation, the surface pressure roughly plateaus in the wake instead of recovering to the stagnation value predicted above, and that asymmetry between the front (attached, pressure-recovering) and rear (separated, plateaued) surface pressure is exactly what produces the nonzero drag that d'Alembert's paradox says shouldn't exist — real cylinders have $C_d \approx 1$–$1.2$ at subcritical Reynolds numbers, nothing like the 0 predicted here.

This is inherently a viscous effect that inviscid potential flow cannot capture. The companion notebook `CFD/CFD Basics/cylinder_cfd/main.ipynb` solves the full unsteady Navier-Stokes equations for this same geometry and shows the separated wake, vortex shedding, and the resulting nonzero drag directly.

## Summary of key results

| Quantity | Computed | Theory |
|---|---|---|
| Min surface $C_p$ (non-lifting) | -3.0000 | -3 |
| $C_d$, all $\Gamma$ (d'Alembert's paradox) | ~0 (< 1e-10) | 0 |
| $C_l$ at $\Gamma=5$ | matches $\Gamma/(RU_\infty)$ to numerical precision | Kutta-Joukowski |
| Stagnation points, subcritical $\Gamma$ | $\sin\theta = -\Gamma/(4\pi RU_\infty)$ | closed-form |